In [42]:
import cv2
import mediapipe as mp
import json
import os
from tqdm import tqdm

mp_pose = mp.solutions.pose

def extract_pose(video_path, out_json_path, vis_out_path=None):
    cap = cv2.VideoCapture(video_path)
    pose = mp_pose.Pose(static_image_mode=False, model_complexity=1, enable_segmentation=False)
    if not cap.isOpened():
        print(f"❌ Failed to open video: {video_path}")
        # Print debug info
        print("💡 Check if file exists:", os.path.exists(video_path))
        print("💡 Absolute path:", os.path.abspath(video_path))
        return
    # Read FPS and frame size right away
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    
    keypoints_all = []
    frame_idx = 0

    if vis_out_path:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        out_vid = cv2.VideoWriter(vis_out_path, fourcc, fps, (width, height))
        # fourcc = cv2.VideoWriter_fourcc(*'MJPG')
        # vis_out_path = vis_out_path.replace(".mp4", ".avi")
        # out_vid = cv2.VideoWriter(vis_out_path, fourcc, fps, (width, height))
        if not out_vid.isOpened():
            print("❌ Failed to open VideoWriter")
            return

    pbar = tqdm(total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(rgb)

        frame_kps = {"frame": frame_idx, "keypoints": {}}

        if results.pose_landmarks:
            for i, lm in enumerate(results.pose_landmarks.landmark):
                frame_kps["keypoints"][str(i)] = {
                    "x": lm.x,
                    "y": lm.y,
                    "z": lm.z,
                    "visibility": lm.visibility
                }

            if vis_out_path:
                mp.solutions.drawing_utils.draw_landmarks(
                    frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)

        keypoints_all.append(frame_kps)

        if vis_out_path:
            out_vid.write(frame)

        frame_idx += 1
        pbar.update(1)

    pbar.close()
    cap.release()
    if vis_out_path:
        out_vid.release()
    pose.close()

    with open(out_json_path, "w") as f:
        json.dump(keypoints_all, f, indent=2)

    print(f"Saved pose data to {out_json_path}")
    if vis_out_path:
        print(f"Saved annotated video to {vis_out_path}")

In [34]:
home_dir = os.path.expanduser("~")
features_dir = os.path.join(home_dir, "badminton/features")
outputs_dir = os.path.join(home_dir, "badminton/outputs")
videos_dir = os.path.join(home_dir, "badminton/videos")

# Create output folders if they don't exist
os.makedirs(features_dir, exist_ok=True)
os.makedirs(outputs_dir, exist_ok=True)

def run_pose_on_clip(clip_name):
    video_path = os.path.join(videos_dir, f"{clip_name}.mp4")
    json_out = os.path.join(features_dir, f"{clip_name}_pose.json")
    vis_out = os.path.join(outputs_dir, f"{clip_name}_pose_annotated.mp4")
    
    extract_pose(
        video_path=video_path,
        out_json_path=json_out,
        vis_out_path=vis_out
    )

In [41]:
# Run pose extraction
run_pose_on_clip("drop_001")


I0000 00:00:1748861154.918059    8783 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1748861154.934403   11584 gl_context.cc:369] GL version: 3.1 (OpenGL ES 3.1 Mesa 24.2.8-1ubuntu1~24.04.1), renderer: D3D12 (NVIDIA GeForce RTX 3070 Ti)
  0%|                                                                                | 0/70 [00:00<?, ?it/s]W0000 00:00:1748861154.992281   11570 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1748861155.025695   11569 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1748861155.039597   11567 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
  9%|██████▏                                    

Pose found on frame 0
Pose found on frame 1
Pose found on frame 2
Pose found on frame 3
Pose found on frame 4
Pose found on frame 5
Pose found on frame 6
Pose found on frame 7
Pose found on frame 8
Pose found on frame 9
Pose found on frame 10


 26%|██████████████████▎                                                    | 18/70 [00:00<00:01, 46.32it/s]

Pose found on frame 11
Pose found on frame 12
Pose found on frame 13
Pose found on frame 14
Pose found on frame 15
Pose found on frame 16
Pose found on frame 17
Pose found on frame 18
Pose found on frame 19
Pose found on frame 20
Pose found on frame 21


 41%|█████████████████████████████▍                                         | 29/70 [00:00<00:00, 49.80it/s]

...

Pose found on frame 44
Pose found on frame 45
Pose found on frame 46
Pose found on frame 47
Pose found on frame 48
Pose found on frame 49
Pose found on frame 50
Pose found on frame 51
Pose found on frame 52
Pose found on frame 53
Pose found on frame 54


 93%|█████████████████████████████████████████████████████████████████▉     | 65/70 [00:01<00:00, 52.85it/s]

Pose found on frame 55
Pose found on frame 56
Pose found on frame 57
Pose found on frame 58
Pose found on frame 59
Pose found on frame 60
Pose found on frame 61
Pose found on frame 62
Pose found on frame 63
Pose found on frame 64
Pose found on frame 65


100%|███████████████████████████████████████████████████████████████████████| 70/70 [00:01<00:00, 49.20it/s]

Pose found on frame 66
Pose found on frame 67
Pose found on frame 68
Pose found on frame 69
Saved pose data to /home/maxlin/badminton/features/drop_001_pose.json
Saved annotated video to /home/maxlin/badminton/outputs/drop_001_pose_annotated.mp4
